# Tabular Classification with CLARYON

Compare multiple classical models (XGBoost, LightGBM, MLP) on tabular data with cross-validation.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer

# Load breast cancer dataset
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=[f"f{i}" for i in range(data.data.shape[1])])
df["label"] = data.target
df.insert(0, "Key", [f"S{i:04d}" for i in range(len(df))])

tmp = tempfile.mkdtemp()
csv_path = Path(tmp) / "breast_cancer.csv"
df.to_csv(csv_path, sep=";", index=False)
print(f"Dataset: {len(df)} samples, {data.data.shape[1]} features")

In [ ]:
import yaml

config_dict = {
    "experiment": {"name": "tabular_demo", "seed": 42, "results_dir": str(Path(tmp) / "Results")},
    "data": {"tabular": {"path": str(csv_path), "label_col": "label", "id_col": "Key", "sep": ";"}},
    "cv": {"strategy": "kfold", "n_folds": 5, "seeds": [42, 123]},
    "models": [
        {"name": "mlp", "type": "tabular"},
    ],
    "evaluation": {"metrics": ["bacc", "auc", "sensitivity", "specificity"]},
    "reporting": {"markdown": True},
}

config_path = Path(tmp) / "tabular.yaml"
with open(config_path, "w") as f:
    yaml.dump(config_dict, f)

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

from claryon.config_schema import load_config
from claryon.pipeline import run_pipeline

config = load_config(config_path)
state = run_pipeline(config)

# Display results
summary_csv = Path(tmp) / "Results" / "metrics_summary.csv"
if summary_csv.exists():
    results = pd.read_csv(summary_csv, sep=";")
    display(results)

In [ ]:
# Load and inspect individual predictions
from claryon.io.predictions import read_predictions

pred_path = Path(tmp) / "Results" / "mlp" / "seed_42" / "fold_0" / "Predictions.csv"
if pred_path.exists():
    preds = read_predictions(pred_path)
    print(f"Predictions shape: {preds.shape}")
    display(preds.head(10))